# **Setup**

In [22]:
# !pip install --upgrade -q langchain langchain-openai langchain-community langchain-core 

In [23]:
from langchain_community.utilities import SQLDatabase
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

import os
from dotenv import load_dotenv
load_dotenv()

True

# **Connect DB**

In [24]:
# 1. Connect to your local PostgreSQL Database
pg_uri = f"postgresql://{os.getenv('DB_USERNAME')}:{os.getenv('DB_PASSWORD')}@{os.getenv('SERVER_NAME')}/{os.getenv('DATABASE_NAME')}?sslmode=require"

db = SQLDatabase.from_uri(pg_uri, 
                        schema=os.getenv('SCHEMA_NAME'), 
                        include_tables=["s6e5_pred_f1pitstops", "s6e3_pred_customerchurn","s5e5_pred_calorieexpenditure"])

# **Connect LLM**

## **LM Studio**

In [25]:
# 2. Point to LM Studio (using modern ChatOpenAI configuration)
llm_lmstudio = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",       # Dummy API key for local routing
    model="ignored-by-local",  # Managed entirely by your active LM Studio GUI model
    temperature=0              # Locked to 0 for strict SQL generation rules
)

## **Google AI**

In [ ]:
# !pip install -q langchain-google-genai

from langchain_google_genai import ChatGoogleGenerativeAI

# 2. Initialize Google's Cloud LLM (Gemini)
# Configuration for Option A (AI Studio API Key setup)
llm_google = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,  # CRITICAL: Keep at 0 to avoid creative/hallucinated SQL syntax
    api_key=os.getenv("GOOGLE_API_KEY") 
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


## **Azure AI**

In [27]:
from langchain_openai import AzureChatOpenAI

# llm_azure = AzureChatOpenAI(
#     azure_endpoint=os.getenv("AZURE_AI_FOUNDRY_ENDPOINT"),
#     api_key = os.getenv("AZURE_OPENAI_API_KEY"),
#     api_version = os.getenv("OPENAI_API_VERSION", "2024-08-01-preview"),
#     azure_deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME", "your-deployment-name"),
#     temperature=0  # CRITICAL: Forces deterministic, non-creative SQL syntax generation
# )

# **Setup Query Pipeline**

In [28]:
# 3. Explicitly construct the SQL generation prompt
sql_prompt = ChatPromptTemplate.from_template(
    """Given the following database schema, write a clean PostgreSQL query that answers the user's question. 
    Return ONLY the raw SQL code. Do not wrap it in markdown code blocks, and do not add explanations.

    Schema:
    {schema}

    Question: {question}
    SQL Query:"""
)

# 4. Construct a completely warning-free functional pipeline (LCEL)
sql_generation_chain = (
    {
        "schema": lambda _: db.get_table_info(), 
        "question": RunnablePassthrough()
    }
    | sql_prompt 
    | llm_google 
    | StrOutputParser()
)

# **Query**

In [29]:
# 5. Run the pipeline
# user_question = "List the unique compounds used in F1 pitstops and their average duration."
user_question = "What's the individual average pitstop duration of each driver?"
# user_question = "What's the most popular contract type among customers who have churned?"

generated_sql = sql_generation_chain.invoke(user_question)

print("--- Generated SQL ---")
print(generated_sql)

# 6. Execute query on DB safely
print("\n--- Execution Result ---")
print(db.run(generated_sql))

--- Generated SQL ---
SELECT driver, AVG(laptime_s) AS average_pitstop_duration
FROM kaggle_playground.s6e5_pred_f1pitstops
GROUP BY driver;

--- Execution Result ---
[('ALB', 92.94699628528971), ('ALE', 92.49253962784286), ('ALG', 92.61325802226587), ('ALO', 92.11576695526693), ('AND', 91.62415263518132), ('ANT', 90.7815107913669), ('ARN', 91.50939389638046), ('BAD', 92.2407359891965), ('BAR', 92.45857548309183), ('BAT', 91.89631172413789), ('BEA', 91.36243603603602), ('BEL', 91.98821793103444), ('BER', 92.09421222296847), ('BIA', 91.78763677712796), ('BOR', 91.4932892376682), ('BOT', 92.99322027649768), ('BRA', 93.58943517241384), ('BRU', 91.74534877384194), ('BUE', 93.90530769230767), ('BUT', 92.04934018126885), ('CAN', 94.05938247282604), ('CAR', 91.52478895184133), ('CEL', 91.2823460750854), ('CHI', 92.14751760797344), ('COL', 91.77963013698631), ('COU', 92.57069083447327), ('D001', 91.55209412550067), ('D002', 91.49611352133044), ('D003', 91.63825708215295), ('D004', 91.311513868